# Part 1: Your First Foundry Agent — SOC Edition

In [3]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

# Auto-detect pre-provisioned environment (created by optional cell in 00-setup)
config_path = Path.cwd().parent / "workshop_config.json"
if not config_path.exists():
    config_path = Path.cwd() / "workshop_config.json"
if config_path.exists():
    _cfg = json.loads(config_path.read_text())
    PROJECT_ENDPOINT = _cfg["PROJECT_ENDPOINT"]
    MODEL_DEPLOYMENT_NAME = _cfg["MODEL_DEPLOYMENT_NAME"]
else:
    raise FileNotFoundError(
        "workshop_config.json not found. Run 00-setup.ipynb in the multi_agent/ or single_agent_skill/ folder first."
    )

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo
   Model: gpt-4.1-mini


---
## Section 1: Your First SOC Foundry Agent (~8 min)

**Foundry Prompt Agents** are server-side agent definitions that combine:
- A **model deployment** (e.g. gpt-4.1)
- **Instructions** (system prompt)
- **Tools** (function calls, web search, file search, etc.)
- **Versioning** — every `create_version()` creates an immutable version

Agents are accessed via the **Responses API** — the same OpenAI-compatible API you know,
but with an `agent_reference` that tells Foundry which agent to use.

In this SOC-focused workshop, we'll build agents that help Security Operations Center analysts
investigate alerts like **impossible travel detections**, **brute force attacks**, and **MFA fatigue**.

In [4]:
# --- Create your first SOC Foundry agent ---
first_agent = project_client.agents.create_version(
    agent_name="01-soc-assistant",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a Security Operations Center (SOC) assistant. "
            "Answer questions about cybersecurity incidents, alert triage, "
            "and Microsoft Sentinel detections concisely and clearly. "
            "If you don't know something, say so."
        ),
    ),
)
print(f"✅ Agent created: {first_agent.name} (version {first_agent.version})")

✅ Agent created: 01-soc-assistant (version 1)


In [8]:
# --- Chat with your SOC agent via the Responses API ---
response = openai_client.responses.create(
    input="What is Microsoft Foundry?",
    extra_body={
        "agent_reference": {"name": first_agent.name, "type": "agent_reference"}
    },
)
print(f"🧑 What is Microsoft Foundry?\n")
print(f"🤖 {response.output_text}")

🧑 What is Microsoft Foundry?

🤖 Microsoft Foundry is a cybersecurity service and collaborative initiative by Microsoft designed to help organizations accelerate their security transformation. It provides expert guidance, best practices, and technical support to improve security posture, streamline security operations, and efficiently adopt Microsoft security solutions such as Microsoft Sentinel, Defender, and Azure security technologies. Foundry often includes workshops, hands-on sessions, and strategic planning tailored to an organization's needs.


#### 🔍 Hallucination vs. Grounded Response

The response above was likely **hallucinated** — gpt-4.1's training data may not include
the latest Microsoft Sentinel documentation or current MITRE mappings.

**Fix: Add an MCP (Model Context Protocol) server.** MCP lets agents call external tools
at inference time. By connecting the [Microsoft Learn docs](https://learn.microsoft.com) via
an MCP server, the agent retrieves **real documentation** instead of guessing.

In [9]:
# --- Grounding with MCP: Microsoft Learn Documentation ---
# Same question, but now the agent can search real docs via MCP.

from azure.ai.projects.models import MCPTool

grounded_agent = project_client.agents.create_version(
    agent_name="01-soc-assistant-grounded",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a Security Operations Center (SOC) assistant. "
            "For any Microsoft related question, use the MCP tool to search Microsoft Learn "
            "Always base your answer on the retrieved documentation and cite sources."
        ),
        tools=[
            MCPTool(
                server_label="microsoft-learn",
                server_url="https://learn.microsoft.com/api/mcp",
                require_approval="never",
            ),
        ],
    ),
)
print(f"✅ Grounded agent created: {grounded_agent.name} (version {grounded_agent.version})")

# Ask the same question — now grounded in real documentation
response = openai_client.responses.create(
    input="What is Microsoft Foundry?",
    extra_body={
        "agent_reference": {"name": grounded_agent.name, "type": "agent_reference"}
    },
)
print(f"\n🧑 What is Microsoft Foundry?\n")
print(f"🤖 {response.output_text}")
print(f"\n✅ Compare this grounded response with the hallucinated one above!")
print(f"   The MCP server gave the agent access to live Microsoft Learn docs.")

✅ Grounded agent created: 01-soc-assistant-grounded (version 2)

🧑 What is Microsoft Foundry?

🤖 Microsoft Foundry is a unified Azure platform-as-a-service designed for enterprise AI operations, model builders, and application development. It combines production-grade infrastructure with user-friendly interfaces so developers can focus on building AI-powered applications rather than managing infrastructure. 

Key features of Microsoft Foundry include:
- Unification of agents, AI models, and tools under a single management grouping with built-in enterprise readiness features such as tracing, monitoring, evaluations, and customizable enterprise setup configurations.
- Streamlined management through unified role-based access control (RBAC), networking, and policies under one Azure resource provider namespace.
- Capability to build and orchestrate multi-agent workflows using SDKs in C# and Python.
- Access to a catalog of over 1,400 tools and integration with Foundry IQ for grounding agent

In [7]:
# --- Multi-turn conversation ---
# Use 'conversation' to maintain state across turns.
# The conversation object tracks history automatically — no need for previous_response_id.

conversation = openai_client.conversations.create()
print(f"Conversation ID: {conversation.id}\n")

# Turn 1
r1 = openai_client.responses.create(
    input="What are the 10 risk factors a SOC analyst should evaluate during an impossible travel investigation?",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": grounded_agent.name, "type": "agent_reference"}},
)
print(f"🧑 Turn 1: What are the 10 risk factors for impossible travel?")
print(f"🤖 {r1.output_text}\n")

# Turn 2 — follows up on the same conversation (context is automatic)
r2 = openai_client.responses.create(
    input="Which of those factors would be most critical for a high-privilege user like a Global Administrator?",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": grounded_agent.name, "type": "agent_reference"}},
)
print(f"🧑 Turn 2: Which factors are most critical for a Global Admin?")
print(f"🤖 {r2.output_text}")

Conversation ID: conv_02030af243513b6500sVm16O50KefGytBqZL6bJVrTdA9uN5dm

🧑 Turn 1: What are the 10 risk factors for impossible travel?
🤖 When a SOC analyst investigates an impossible travel alert (where sign-ins occur from geographically distant locations within an unrealistically short time), evaluating certain risk factors helps determine if the alert is legitimate or a false positive. Although Microsoft documentation does not list exactly "10 risk factors" in one place, it provides investigative guidance that aligns closely with evaluating key risk factors during an impossible travel investigation.

From the official Microsoft resources on investigating such anomalies (especially in Microsoft Sentinel and Microsoft Entra ID protection), here are 10 risk factors and considerations a SOC analyst should evaluate during an impossible travel investigation:

1. **User's Known and Common Locations:**  
   Check if the locations involved in the logins are among the user's commonly known or

> 💡 **Check it out**: Open the [Foundry Portal](https://ai.azure.com) → your project → **Tracing**.
> You should see server-side traces for each `responses.create()` call with an `agent_reference`.

---
Proceed to `02-tools.ipynb`.